In [2]:
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from typing import Annotated
import os


@tool(description="获取天气信息")
def getWeather(city: Annotated[str, "查询哪个城市的天气"] = "北京") -> str:
    return f"{city}的天气是晴天"


client = ChatOpenAI(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
)

# 内部使用langgraph创建的一个agent
agent = create_agent(
    model=client,
    tools=[getWeather]
)

# response = agent.invoke({
#     "messages": [HumanMessage("今天天气怎么样")]
# })
#
# print(response["messages"][-1].content)

# messages模式
stream = agent.stream(
    input={"messages": [HumanMessage("今天天气怎么样")]},
    stream_mode="messages"
)

# 每个chunk是 (消息块, 元数据)
for msg_chunk, meta in stream:
    # 只打印文本内容，工具调用阶段msg_chunk.content是空
    if msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)

北京的天气是晴天今天北京的天气是**晴天** ☀️，天气不错，适合出门活动。如果想了解具体温度或空气质量，可以再告诉我。